In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset


In [3]:
MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"
DATASET_ID = "yahma/alpaca-cleaned"


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NF4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,    # nested quantization saves a bit more memory
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:178: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [42]:
from transformers import pipeline

In [ ]:
from rouge_score import rouge_scorer

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
dataset = load_dataset(DATASET_ID)

In [44]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [49]:
generator("Tell me about artificial intelligence", max_new_tokens=10)

Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Tell me about artificial intelligence.\n\nArtificial Intelligence\n\nThe term “'}]

In [13]:
dataset['train'][210]

{'output': 'Standalone question: What are some effective ways to practice speaking a foreign language with native speakers?',
 'input': 'How can I practice speaking with native speakers?',
 'instruction': "Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question.\nChat History:\nUser: What's the best way to learn a new language?\nAI: The most effective way to learn a new language is by practicing regularly, immersing yourself in the language, using it in real-life situations, and studying with various resources like books, apps, and language partners.\nUser: How long does it typically take to become fluent?\nAI: That depends on several factors, including how often one is practicing the language, what resources are available and whether or not the person is immersed."}

In [14]:
tokenizer(dataset['train'][210]['instruction'], return_tensors="pt")

{'input_ids': tensor([[15423,   260,  1695,  6634,   284,   253,  1066,   614,  1962,    28,
           298, 33166,   260,  1066,   614,  1962,   288,   325,   253, 36947,
          1962,    30,   198, 40558,  3760,    42,   198, 11126,    42,  1812,
           506,   260,  1450,   970,   288,   835,   253,   725,  1789,    47,
           198, 13701,    42,   378,   768,  2430,   970,   288,   835,   253,
           725,  1789,   314,   411,  8145,  5578,    28, 35651,  2747,   281,
           260,  1789,    28,  1015,   357,   281,  1345,    29,  3460,  4403,
            28,   284,  4836,   351,  1461,  1952,   702,  2905,    28,  9527,
            28,   284,  1789,  4430,    30,   198, 11126,    42,  1073,   986,
          1072,   357,  3431,  1188,   288,  1438, 30634,    47,   198, 13701,
            42,  1848,  5552,   335,  1545,  2212,    28,  1285,   638,  1129,
           582,   314,  8145,   260,  1789,    28,   732,  1952,   359,  1770,
           284,  1991,   355,   441,  

In [17]:
output = model(**tokenizer(dataset['train'][210]['instruction'], return_tensors="pt").to(model.device))

/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:204: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:314: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [19]:
from pprint import pprint

pprint(tokenizer.decode(output.logits.argmax(dim=-1)[0]))

(' controvers fact information between the few- question, how- the question up '
 'question in make more question question.\n'
 '\n'
 'bot\n'
 '\n'
 '\n'
 '1 Hi is the difference way to get a new language?\n'
 'User: I best effective way to learn a new language is to immersion it. and '
 'yourself in the language, and it in real-life situations, and seeking '
 'grammar a resources. books, courses, and classes courses.\n'
 'User: How can does it take take to learn fluent in\n'
 'AI: It depends on several factors, including the much you practices exposed, '
 'language, the level are being, how one not one person is motivated in\n')


In [33]:
text = "Hugging Face makes NLP so much easier."
inputs = tokenizer(text, return_tensors="pt")

In [39]:
inputs

{'input_ids': tensor([[   56, 19712,  8182,  2022, 34880,   588,  1083,  4285,    30]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [41]:
model(**inputs)

RuntimeError: Placeholder storage has not been allocated on MPS device!

In [40]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

RuntimeError: Placeholder storage has not been allocated on MPS device!

In [35]:
output = model(**tokenized, max_new_tokens=512)

/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:204: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/Users/rasheddoha/Documents/Cursor Project/lora-peft-finetuning/.venv/lib/python3.13/site-packages/bitsandbytes/backends/default/ops.py:314: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [32]:
output.logits.shape

torch.Size([1, 8, 49152])

In [28]:
tokenizer.decode(output.logits.argmax(dim=-1))

[' controvers ( a branch that is used1']

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model = AutoModelForCausalLM.from_pretrained("alexsherstinsky/Mistral-7B-v0.1-sharded")
tokenizer = AutoTokenizer.from_pretrained("alexsherstinsky/Mistral-7B-v0.1-sharded")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 